# Transformer (COMING SOON)

In this section, we will generate forecasts of EEG activity using a Transformer-based model. The VARIMA, DMD, and TCN models primarly focus on modeling short-term temporal correlations within a system and struggle to capture the global dynamics present within long-term time windows. Transformers, on the other hand, excel at modeling the long-range dependencies between samples of a sequence—making them a promising approach to characterize and forecast the multi-scale temporal structures present in neural activity.    

## Background

The Transformer architecture was introduced in the seminal paper by Vaswani et al. (2017), which demonstrated that attention mechanisms alone were sufficient for solving complex sequence modeling and transduction problems. This architecture represented a fundamental shift in deep learning by dispensing with the recursive constraints of traditional models (e.g., recurrent networks). By processing entire sequences simultaneously rather than step by step, the Transformer enables the modeling of lengthy sequences that were previously computationally prohibitive.

Transformers have exhibited unparalleled success within the field of Natural Language Processing (NLP). In tasks such as machine translation, this architecture serves as a core component of cutting-edge Large Language Models (LLMs) that effectively capturing the semantic relationships between words and concepts, irrespective of there distance within an excerpt.

### Queries, Keys, and Values

At the core of the Transformer is the **self-attention mechanism**, which allows the model to decide how much each position in a sequence should "look at" every other position when building its representation.

To build intuition, consider a library search: you type a **query** describing what you want, the library matches it against the **keys** (catalog index entries) of every book on the shelf, and the books with the closest-matching keys are retrieved—their **values** (the actual content) are then blended together in proportion to how well they matched your query. Self-attention works the same way, except every position in the sequence simultaneously acts as both a searcher (submitting a query) *and* a book (advertising a key and holding a value).

Concretely, given an input matrix $X \in \mathbb{R}^{T \times d_{\text{model}}}$, where $T$ is the sequence length and $d_{\text{model}}$ is the embedding dimension, three separate linear projections produce the query, key, and value matrices:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

where $W^Q, W^K, W^V \in \mathbb{R}^{d_{\text{model}} \times d_k}$ are learned weight matrices and $d_k$ is the projection dimension. Because all three matrices are derived from the same input $X$, the mechanism is called **self**-attention—each position queries and is queried by every other position in the same sequence.

### Scaled Dot-Product Attention

Attention scores are computed by measuring the similarity between each query and every key via a scaled dot product. The resulting scores are normalized with a softmax so they sum to one, then used to form a weighted sum of the values:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The division by $\sqrt{d_k}$ is a stabilizing trick: when $d_k$ is large the dot products tend to grow large in magnitude, pushing the softmax into regions where its gradients nearly vanish. Scaling counters this. The softmax output is an attention weight matrix $A \in \mathbb{R}^{T \times T}$, where entry $A_{ij}$ encodes how strongly time step $i$ attends to time step $j$.

Rather than performing a single attention operation, the Transformer employs **multi-head attention**, running $h$ independent attention operations in parallel across different learned subspaces and concatenating their outputs:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)\,W^O, \quad \text{head}_i = \text{Attention}(QW_i^Q,\; KW_i^K,\; VW_i^V)$$

where $W^O \in \mathbb{R}^{hd_k \times d_{\text{model}}}$ is a learned output projection. Each head is free to specialize in a different kind of relationship—one head might track short-range structure while another captures longer-range dependencies—giving the model far richer representational capacity than a single attention operation.

### Autoregressive Prediction

The original Transformer follows an **encoder–decoder** design. The encoder reads the full input sequence and produces a context representation via self-attention. The decoder then generates the output sequence one step at a time, using **cross-attention** to consult that context at each step.

Cross-attention operates the same way as self-attention, but the queries and keys/values come from *different* sequences. Specifically, the decoder's current output supplies the queries ($Q$), while the encoder's context supplies the keys and values ($K$, $V$):

$$\text{CrossAttention}(Q_\text{dec},\, K_\text{enc},\, V_\text{enc}) = \text{softmax}\!\left(\frac{Q_\text{dec}\,K_\text{enc}^\top}{\sqrt{d_k}}\right)V_\text{enc}$$

This lets each decoder position ask: *"which parts of the input sequence are most relevant to what I am about to predict?"* and retrieve the corresponding encoder context.

During training, the decoder also applies **masked self-attention** over its own outputs: future positions are hidden by setting their attention scores to $-\infty$ before the softmax, so the model can only attend to steps it has already generated. This ensures the model learns to predict each step from the past alone—without peeking at the future.

At inference, predictions are generated **autoregressively**: the model predicts step $t+1$, appends it to the decoder input, predicts step $t+2$, and so on until the target horizon $H$ is reached:

$$\hat{x}_{t+h} = f_\theta\!\left(x_1, \ldots, x_t,\; \hat{x}_{t+1}, \ldots, \hat{x}_{t+h-1}\right), \quad h = 1, \ldots, H$$

Because each generated step is fed back as input for the next, any prediction error accumulates over the horizon—a phenomenon known as **exposure bias**. This compounding makes long-horizon forecasting substantially harder than single-step prediction, and is one of the primary motivations for the time series adaptations discussed below.

### From NLP to Time Series Forecasting

Adapting the Transformer from discrete language tokens to continuous-valued time series requires several modifications.

**Input representation.** In NLP, tokens are drawn from a finite vocabulary and embedded via a lookup table. Time series observations are continuous, so a learned linear projection is used instead to map each time step's values into the model dimension. For multivariate recordings like EEG, the input at each step is a vector $\mathbf{x}_t \in \mathbb{R}^C$ across $C$ channels.

**Positional encoding.** Because self-attention is permutation-invariant—it has no built-in notion of order—a positional encoding is added to each embedding to inject information about where in the sequence a sample falls. NLP models use sinusoidal encodings or learned absolute positions. Time series models often enrich this with domain-specific temporal features such as time-of-day, frequency band, or trial index.

**Efficient attention for long sequences.** Computing $QK^\top$ requires a dot product between every pair of time steps, producing an attention matrix of size $T \times T$. Both the computation and memory therefore scale as $\mathcal{O}(T^2)$—doubling the sequence length quadruples the cost. For short sentences this is acceptable, but for long high-frequency signals like EEG (e.g., $T = 1000$ steps at 256 Hz), the attention matrix alone contains $10^6$ entries, making full attention computationally prohibitive.

Several architectures address this:

- **FEDformer** (Zhou et al., 2022) performs attention in the frequency domain using a sparse Fourier or wavelet decomposition. By attending over spectral components rather than individual time steps, it efficiently captures the global periodicities and trends that are particularly prominent in neural oscillations.

- **PatchTST** (Nie et al., 2023) divides the input into non-overlapping patches and treats each patch as a token, dramatically shortening the effective sequence length and improving long-horizon accuracy.

- **Direct multi-step prediction head.** Rather than decoding one step at a time as in NLP, many time series Transformers attach a linear projection head to the encoder output that maps the full context representation to all $H$ future steps simultaneously, sidestepping the exposure bias inherent in autoregressive decoding.

- **Informer** (Zhou et al., 2021) introduces *ProbSparse* attention. The key observation is that in practice, each query's attention distribution is highly uneven: a small number of keys dominate and the rest receive near-zero weight. Informer identifies the most *informative* queries—those with the most peaked distributions—by scoring each query $q_i$ against a random sample of $\ln T$ keys:

$$\bar{M}(q_i, K) = \max_{j} \left\{\frac{q_i k_j^\top}{\sqrt{d_k}}\right\} - \frac{1}{T}\sum_{j=1}^{T} \frac{q_i k_j^\top}{\sqrt{d_k}}$$

  A large $\bar{M}$ means the query's attention is concentrated (informative); a small value means it is nearly uniform (uninformative). Only the top-$u = c \ln T$ queries by this score are selected to participate in full attention, while all remaining queries default to the mean of $V$:

$$\text{ProbSparseAttn}(\bar{Q}, K, V) = \text{softmax}\!\left(\frac{\bar{Q}K^\top}{\sqrt{d_k}}\right)V$$

  where $\bar{Q} \in \mathbb{R}^{u \times d_k}$ is the selected query subset. This reduces complexity to $\mathcal{O}(T \log T)$ in both time and memory.

In this tutorial, we implement the **Informer**, whose ProbSparse attention mechanism makes it well-suited to the long, high-frequency nature of EEG recordings.

## Implementing

With the theoretical foundation in place, we can now see the Informer in action. The sections below build the model from the ground up: first the attention mechanisms, then the multi-head wrapper and positional encoding, then the encoder layer with its distilling operation, and finally the full `TimeSeriesTransformer` that ties everything together.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

### Attention Mechanisms

We implement two attention functions side by side. `scaled_dot_product_attention` is the standard $\mathcal{O}(T^2)$ formulation and serves as a reference. `ProbSparseAttention` is the Informer's replacement: it scores every query cheaply using a sample of keys, selects only the most informative ones, and runs full attention exclusively on that subset — reducing complexity to $\mathcal{O}(T \log T)$.

In [ ]:
# ── Traditional Scaled Dot-Product Attention (reference) ─────────────────────
# Computes full O(T²) attention — every query attends to every key.
def scaled_dot_product_attention(
    Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    d_k: int = Q.size(-1)
    # Build the T×T pairwise similarity matrix
    scores: torch.Tensor = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    # Normalize to a probability distribution and retrieve values
    attn: torch.Tensor = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V)


# ── ProbSparse Attention (Informer) ──────────────────────────────────────────
# Only the top-u most informative queries run full attention; the rest default
# to mean(V), reducing complexity to O(T log T).
class ProbSparseAttention(nn.Module):

    def __init__(self, factor: int = 5, dropout: float = 0.1):
        super().__init__()
        self.factor: int = factor   # sampling constant c; selects u = c·ln(T) queries
        self.dropout = nn.Dropout(dropout)

    def _sparsity_score(
        self, Q: torch.Tensor, K: torch.Tensor, sample_k: int
    ) -> torch.Tensor:
        """Score each query: large score → peaked distribution → informative."""
        L_K: int = K.size(2)
        # Sample a random subset of keys to approximate the score cheaply
        idx: torch.Tensor = torch.randint(L_K, (sample_k,), device=Q.device)
        K_sample: torch.Tensor = K[:, :, idx, :]                            # (B, H, sample_k, D)
        # Dot products between every query and the sampled keys
        QK: torch.Tensor = torch.matmul(Q, K_sample.transpose(-2, -1))      # (B, H, L_Q, sample_k)
        # M = max − mean: high when attention is concentrated on a few keys
        return QK.max(dim=-1).values - QK.mean(dim=-1)                      # (B, H, L_Q)

    def forward(
        self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, H, L_Q, D = Q.shape
        L_K: int = K.size(2)
        scale: float = D ** -0.5

        # Select u = factor·ln(L_K) top queries, capped at L_Q
        u: int = min(self.factor * math.ceil(math.log(L_K)), L_Q)
        M: torch.Tensor = self._sparsity_score(Q, K, sample_k=u)
        top_idx: torch.Tensor = M.topk(u, dim=-1).indices                   # (B, H, u)

        # Gather only the u selected queries for full attention
        Q_sparse: torch.Tensor = Q.gather(
            dim=2, index=top_idx.unsqueeze(-1).expand(-1, -1, -1, D)
        )                                                                    # (B, H, u, D)

        # Full attention over selected queries only — O(u·L_K) vs O(L_Q·L_K)
        scores: torch.Tensor = torch.matmul(Q_sparse, K.transpose(-2, -1)) * scale
        if mask is not None:
            scores = scores.masked_fill(mask[:, :, :u, :L_K] == 0, float('-inf'))
        attn: torch.Tensor = self.dropout(F.softmax(scores, dim=-1))
        attended: torch.Tensor = torch.matmul(attn, V)                      # (B, H, u, D)

        # Non-selected queries default to the uniform-attention baseline: mean(V)
        out: torch.Tensor = V.mean(dim=2, keepdim=True).expand(B, H, L_Q, D).clone()
        out.scatter_(dim=2, index=top_idx.unsqueeze(-1).expand(-1, -1, -1, D), src=attended)
        return out, attn

### Multi-Head Attention and Positional Encoding

`MultiHeadAttention` wraps `ProbSparseAttention` in the standard multi-head projection scheme: it linearly projects the input into $h$ separate Q, K, V subspaces, runs ProbSparse attention in each head in parallel, then merges and re-projects the results. `PositionalEncoding` adds a fixed sinusoidal signal to each embedded time step so the model knows the order of the sequence.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Runs ProbSparse attention across h independent learned subspaces."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads: int = n_heads
        self.d_k: int = d_model // n_heads   # dimension per head

        # Separate weight matrices produce Q, K, V from the same input
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model)    # merges all heads back to d_model
        self.attn = ProbSparseAttention(dropout=dropout)
        self.dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """Reshape (B, T, d_model) → (B, n_heads, T, d_k) for parallel head computation."""
        B, T, _ = x.shape
        return x.view(B, T, self.n_heads, self.d_k).transpose(1, 2)

    def forward(
        self, Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # Project input into Q, K, V and split across heads
        q: torch.Tensor = self._split_heads(self.W_Q(Q))
        k: torch.Tensor = self._split_heads(self.W_K(K))
        v: torch.Tensor = self._split_heads(self.W_V(V))
        # Attention runs simultaneously across all heads
        ctx, _ = self.attn(q, k, v, mask=mask)
        # Merge heads and project back to d_model
        ctx = ctx.transpose(1, 2).contiguous().view(Q.size(0), Q.size(1), -1)
        return self.dropout(self.W_O(ctx))


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        # Pre-compute the encoding table: alternating sin/cos across model dimensions
        pe: torch.Tensor = torch.zeros(max_len, d_model)
        pos: torch.Tensor = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div: torch.Tensor = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Add the positional signal to each embedded time step
        return self.dropout(x + self.pe[:, :x.size(1)])

### Encoder Layer

Each `EncoderLayer` combines three operations. First, ProbSparse multi-head self-attention with a residual connection lets each time step gather context from informative positions across the sequence. Second, a position-wise feed-forward network applies a non-linear transformation independently at each step. Third, a **distilling** operation — a 1-D convolution followed by max-pooling — halves the sequence length, progressively compressing the temporal context as it passes through successive encoder layers. This cascading reduction is a key feature of the Informer: it trades some temporal resolution for dramatically lower memory and compute at deeper layers.

In [ ]:
class EncoderLayer(nn.Module):
    """Informer encoder layer: ProbSparse self-attention + feed-forward + distilling."""

    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)

        # Position-wise feed-forward: expands to d_ff then projects back
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        # Layer norms stabilize training after each sub-layer
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # Distilling: ELU-activated conv + max-pool halves the sequence length
        self.distil = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.ELU(),
            nn.MaxPool1d(kernel_size=2, stride=2),
        )

    def forward(
        self, x: torch.Tensor, mask: torch.Tensor | None = None
    ) -> torch.Tensor:
        # Self-attention sub-layer with residual connection
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))
        # Feed-forward sub-layer with residual connection
        x = self.norm2(x + self.ff(x))
        # Distilling: (B, T, d_model) → (B, T//2, d_model)
        x = self.distil(x.transpose(1, 2)).transpose(1, 2)
        return x

### TimeSeriesTransformer

`TimeSeriesTransformer` assembles the full model. A linear input projection maps the raw EEG channel values at each time step into the model's embedding space, after which positional encodings are added. The signal then passes through the stack of encoder layers, each halving the sequence length via distilling. Finally, a global average pool reduces the compressed sequence to a single context vector, which a linear output head maps directly to all `pred_len` forecast steps across all channels — the direct multi-step prediction strategy described in the Background section.

In [ ]:
class TimeSeriesTransformer(nn.Module):
    """
    Informer-based Transformer for multivariate EEG forecasting.

    Args:
        n_channels:   Number of EEG channels (input and output features).
        d_model:      Embedding dimension throughout the model.
        n_heads:      Number of parallel attention heads.
        n_enc_layers: Encoder depth; each layer halves the sequence length.
        d_ff:         Inner dimension of the feed-forward sub-layers.
        pred_len:     Number of future time steps to forecast.
        dropout:      Dropout probability applied throughout.
    """

    def __init__(
        self,
        n_channels: int,
        d_model: int = 64,
        n_heads: int = 4,
        n_enc_layers: int = 2,
        d_ff: int = 256,
        pred_len: int = 32,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.pred_len: int = pred_len
        self.n_channels: int = n_channels

        # Project raw channel values into the model's embedding space
        self.input_proj = nn.Linear(n_channels, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # Stack of encoder layers; sequence length shrinks by 2× per layer
        self.encoder = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout)
            for _ in range(n_enc_layers)
        ])

        # Direct multi-step head: context vector → all pred_len steps × channels
        self.output_proj = nn.Linear(d_model, pred_len * n_channels)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Embed and encode the input sequence through all encoder layers."""
        z: torch.Tensor = self.pos_enc(self.input_proj(x))
        for layer in self.encoder:
            z = layer(z)
        return z                                            # (B, T', d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, C) — input EEG window.
        Returns:
            (B, pred_len, C) — forecasted values.
        """
        z: torch.Tensor = self.encode(x)                   # (B, T', d_model)
        # Global average pool collapses the compressed sequence to one context vector
        z_ctx: torch.Tensor = z.mean(dim=1)                # (B, d_model)
        # Project to all forecast steps simultaneously and reshape
        out: torch.Tensor = self.output_proj(z_ctx)        # (B, pred_len * n_channels)
        return out.view(x.size(0), self.pred_len, self.n_channels)